# OpenEnv Reconcile-GST2B-Env: Demo Notebook

**Multi-turn enterprise compliance workflow RL on OpenEnv.** Indian GST Input-Tax-Credit reconciliation. ~14M businesses affected monthly.

This notebook lets you:

1. Verify the environment imports cleanly
2. Run an oracle episode to see the 16-verb tool surface in action
3. Inspect the n=5 trained-checkpoint audit (n=5 mean composite reward 0.305) without re-training
4. View the embedded loss curve, baseline-comparison, training-progression, and GRPO-curve plots

Full training: `train_sft_warmstart.py` ran 31 min on A100 SXM4-80GB Day 1, followed by 100 GRPO steps with Tier 2c length-shaping. Re-running on Colab T4 would take 3-4 hours and is not required to evaluate this submission. Pre-trained checkpoint artifacts are committed under `envs/reconcile_gst2b_env/data/`.

- Live HF Space: https://huggingface.co/spaces/akashkathole/reconcile_gst2b_env
- Repo: https://github.com/akashkathole7/OpenEnv (branch: `scaffold/reconcile-gst2b`)
- Headline: trained Qwen3-4B SFT + P3 GRPO, n=5 mean **0.305** (above prompted Qwen2.5-3B 0.18, below red-team ceiling 0.45)
- 5 documented failure modes including FM4 (audit-OOD trap chain) and FM5 (reward-landscape inversion). See [LESSONS_LEARNED.md](https://github.com/akashkathole7/OpenEnv/blob/scaffold/reconcile-gst2b/envs/reconcile_gst2b_env/LESSONS_LEARNED.md) §1.

Estimated total runtime on free Colab T4: under 5 minutes.

## Setup: install dependencies and clone repo

In [ ]:
!pip install -q transformers trl peft accelerate datasets bitsandbytes pydantic networkx plotly matplotlib
!git clone -b scaffold/reconcile-gst2b https://github.com/akashkathole7/OpenEnv.git
%cd OpenEnv
!pip install -q -e .

## Verify environment imports

In [ ]:
import sys
sys.path.insert(0, 'src')
sys.path.insert(0, '.')

from envs.reconcile_gst2b_env.server.reconcile_gst2b_environment import ReconcileGST2BEnvironment

env = ReconcileGST2BEnvironment()
obs = env.reset(seed=42, mode='warmup')
print(f'Environment OK. Step budget: {obs.step_budget}, Invoices remaining: {obs.invoices_remaining_count}')

## Run an oracle episode (the upper-bound reference policy)

The oracle reads the env's hidden ground truth (only available to scripts, never to the trained agent) and emits a sequence of verbs that scores well. This is the upper-bound reference: a healthy trained policy should approach this. The 4-component reward breakdown shows R1 (label macro-F1), R2 (ITC delta accuracy), R3 (Rule 36(4) per-supplier compliance), R4 (step efficiency).

In [ ]:
from envs.reconcile_gst2b_env.scripts._policies import (
    oracle_heuristic_policy,
    populate_context_from_result,
)
import random

env = ReconcileGST2BEnvironment()
obs = env.reset(seed=9502, mode='warmup')
context = {'_env': env}
rng = random.Random(9502)
trajectory = []

while not obs.done and len(trajectory) < 50:
    action = oracle_heuristic_policy(obs, rng, context)
    trajectory.append({'verb': action.verb, 'invoice_id': action.payload.get('invoice_id', '')})
    obs = env.step(action)
    populate_context_from_result(context, obs.last_tool_result or {})

print(f'Episode complete. Trajectory length: {len(trajectory)}')
print(f'Reward breakdown: {env.state.reward_breakdown}')
print()
print('First 5 actions:')
for a in trajectory[:5]:
    print(f'  {a}')
print()
print('Last 5 actions:')
for a in trajectory[-5:]:
    print(f'  {a}')

## Inspect the trained-checkpoint audit (no GPU required)

The Day 1 on-site SFT (375 steps) + P3 GRPO (100 steps with Tier 2c length-shaping) checkpoint was audited on 5 held-out seeds (9030 to 9034) at GRPO-matching sampling (T=0.7, top_p=0.95, top_k=20) with `tools=` enabled. The full audit JSON is in `data/audit_grpo_p3_F_n5.json`.

In [ ]:
import json

audit = json.load(open('envs/reconcile_gst2b_env/data/audit_grpo_p3_F_n5.json'))

totals = [r['reward_breakdown'].get('total', 0.0) for r in audit['rollouts']]
mean_total = sum(totals) / len(totals)

print('Trained Qwen3-4B SFT + P3 GRPO audit on heldout seeds 9030 to 9034:')
print(f'  n=5 mean composite reward: {mean_total:.4f}')
print()
print('5-metric trajectory-quality audit (averaged):')
for k, v in audit['metrics_avg'].items():
    print(f'  {k}: {v:.4f}')
print()
print('Per-seed totals:')
for r in audit['rollouts']:
    rb = r['reward_breakdown']
    print(f"  seed {r['seed']}: total={rb.get('total', 'n/a'):.4f} "
          f"(R1={rb.get('R1', 'n/a')}, R2={rb.get('R2', 'n/a')}, "
          f"R3={rb.get('R3', 'n/a')}, R4={rb.get('R4', 'n/a')}), "
          f"steps={r['total_steps']}, term={r['termination']}")

## Embedded plots (committed under `data/figures/`)

In [ ]:
from IPython.display import Image, Markdown, display
import os

figures = [
    ('day1_baseline_comparison.png',
     'Day 1 baseline comparison: trained Qwen3-4B SFT + P3 GRPO (purple, 0.305) vs oracle, 6 red-team attacks, prompted Qwen2.5-3B baseline (0.18), and 0.45 red-team ceiling.'),
    ('day1_training_progression.png',
     'Day 1 training progression: SFT step 350 (0.314) vs step 375 final (0.280). Empirical proof of FM5 reward-landscape inversion: the MORE-trained checkpoint scores LOWER because it shifts more seeds from the cheap query_only attack (0.353) into marking trajectories (0.17 to 0.26).'),
    ('grpo_reward_curve.png',
     'P3 GRPO with Tier 2c length-shaping bonus. Top pane: train reward over 100 steps, 10-step rolling mean trends from ~0.18 to ~0.23. Bottom pane: visual contrast between the buggy fresh-context eval callback (flat 0.354 across all 6 logged steps) and the true post-training audit (0.305). The gap is the empirical measure of FM4 audit-OOD bug.'),
    ('three_scales_reward.png',
     'Pre-onsite Qwen3-0.6B + LoRA + GRPO on Kaggle T4: 150-step eval composite pins at 0.353 (within 0.004 of the query_only red-team attack). Pure GRPO without SFT warm-start drifts into attack-signature territory at this scale.'),
    ('three_scales_components.png',
     'Per-component R1/R2/R3/R4 breakdown showing the defense-in-depth reward contract: each red-team attack pins a different subset of components, and no single-component exploit clears the 0.45 composite ceiling.'),
    ('loss_curve.png',
     'Pre-onsite Qwen3-0.6B 10-step SFT smoke run loss trajectory.'),
]

for fname, caption in figures:
    path = f'envs/reconcile_gst2b_env/data/figures/{fname}'
    if os.path.exists(path):
        display(Markdown(f'### {fname}\n\n{caption}'))
        display(Image(path))
    else:
        print(f'[missing] {path}')

## Re-run full training (optional, requires A100 or longer wait on T4)

The full training is NOT executed in this notebook (would take 3-4 hours on free Colab T4 and is not required to evaluate this submission). The exact commands that ran on the on-site A100 SXM4-80GB are below for reference. Pre-trained checkpoint artifacts are committed under `envs/reconcile_gst2b_env/data/`.

**Phase 2 SFT (375 optimizer steps, 31 minutes wall on A100):**

```bash
PYTHONPATH=src:envs python -m envs.reconcile_gst2b_env.scripts.train_sft_warmstart \
    --input-jsonl envs/reconcile_gst2b_env/data/sft_trajectories.jsonl \
    --output-dir  envs/reconcile_gst2b_env/data/sft_checkpoint \
    --model       Qwen/Qwen3-4B \
    --epochs      1 \
    --batch-size  1 \
    --grad-accum  8 \
    --learning-rate 2e-5 \
    --precision   bf16 \
    --max-seq-length 2048 \
    --logging-steps 5 \
    --save-steps 50
```

**LoRA merge (required before GRPO; the GRPO trainer loads via `AutoModelForCausalLM.from_pretrained`):**

```python
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base = AutoModelForCausalLM.from_pretrained('Qwen/Qwen3-4B', torch_dtype='auto', device_map='auto')
merged = PeftModel.from_pretrained(
    base, 'envs/reconcile_gst2b_env/data/sft_checkpoint/final'
).merge_and_unload()
merged.save_pretrained('envs/reconcile_gst2b_env/data/sft_checkpoint/merged')
AutoTokenizer.from_pretrained('Qwen/Qwen3-4B').save_pretrained(
    'envs/reconcile_gst2b_env/data/sft_checkpoint/merged'
)
```

**Phase 3 GRPO (100 steps with Tier 2c length-shaping, 100 minutes wall on A100):**

```bash
PYTHONPATH=src:envs python -m envs.reconcile_gst2b_env.scripts.train_grpo_real \
    --model envs/reconcile_gst2b_env/data/sft_checkpoint/merged \
    --total-steps 100 \
    --output-dir envs/reconcile_gst2b_env/data/grpo_p3_checkpoint
```

**5-metric rollout audit on the post-GRPO checkpoint:**

```bash
PYTHONPATH=src:envs python -m envs.reconcile_gst2b_env.scripts.audit_sft_rollout_quality \
    --checkpoint envs/reconcile_gst2b_env/data/grpo_p3_checkpoint/merged \
    --seeds 9030 9031 9032 9033 9034 \
    --max-steps 50 \
    --temperature 0.7 --top-p 0.95 --top-k 20 \
    --use-tools
```

Note: the audit script requires the `--use-tools` flag to pass the 16-verb schema list to `apply_chat_template`. Without it, the model emits zero marks across n=5 seeds (apparent collapse). This is documented as Failure Mode 4 in `LESSONS_LEARNED.md` §1.

## Where to go next

- **Live demo:** [HF Space](https://huggingface.co/spaces/akashkathole/reconcile_gst2b_env). Default landing tab is Tab 4 (Baseline Comparison) showing the trained 0.305 bar live. Tab 3 is the 3D circular-trading-ring viewer.
- **Research findings:** [LESSONS_LEARNED.md §1](https://github.com/akashkathole7/OpenEnv/blob/scaffold/reconcile-gst2b/envs/reconcile_gst2b_env/LESSONS_LEARNED.md). 5 documented failure modes:
  - 3 pre-onsite Qwen3-scale modes (0.6B pins R1/R2, 1.7B entropy collapse, 4B over-query)
  - FM4 (audit-OOD trap chain): 4 sequential bugs in audit infrastructure each fabricated a different false collapse signature. Empirically generalizes across SFT, GRPO, and eval code paths (the buggy fresh-context eval callback reported flat 0.354 across all 100 GRPO steps).
  - FM5 (reward-landscape inversion): under arithmetic-composite + R3-R4 saturation, marking trajectories (Mode A) score lower than the cheap query_only attack (Mode B). The MORE-trained checkpoint at step 375 scores LOWER than at step 350 because more seeds shifted from Mode B (0.353) into Mode A (0.17 to 0.26). Partially mitigated by P3 Tier 2c length-shaping bonus (+0.025 lift). Full bimodal flip is post-hackathon agenda.
- **Long-form narrative:** [BLOG.md](https://github.com/akashkathole7/OpenEnv/blob/scaffold/reconcile-gst2b/envs/reconcile_gst2b_env/BLOG.md) §6 closing block has the on-site Day 1 outcome story.
- **Round 2 problem statement:** [ROUND2_PROBLEM_STATEMENT.md](https://github.com/akashkathole7/OpenEnv/blob/scaffold/reconcile-gst2b/envs/reconcile_gst2b_env/ROUND2_PROBLEM_STATEMENT.md) maps the env design to the rubric criteria.
- **Audit script:** [`scripts/audit_sft_rollout_quality.py`](https://github.com/akashkathole7/OpenEnv/blob/scaffold/reconcile-gst2b/envs/reconcile_gst2b_env/scripts/audit_sft_rollout_quality.py) is the diagnostic tool that produced the 0.305 headline. Supports `--use-tools`, multi-turn accumulation, GRPO-matching sampling, off-vocab guard.
- **Red-team battery:** [`tests/envs/test_reconcile_gst2b_reward_hacking.py`](https://github.com/akashkathole7/OpenEnv/blob/scaffold/reconcile-gst2b/tests/envs/test_reconcile_gst2b_reward_hacking.py) enforces 6 attacks all under 0.45 in CI. Any reward change that lifts any attack above the ceiling fails the build.

Submission context: solo finalist Aakash Kathole, Meta x Scaler Hackathon Bangalore 2026, Round 2 Theme #3.1 Professional Tasks / World Modeling, sub-theme Scaler AI Labs Multi-App RL Environment for Enterprise Workflows.